<a href="https://colab.research.google.com/github/patriciasandagorda/WM/blob/main/Scraping_lotes_ps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install requests beautifulsoup4 playwright playwright-stealth

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    capture_output=True, text=True
)
print(result.stdout or "Chromium ya instalado.")
if result.returncode != 0:
    print("STDERR:", result.stderr)

Chromium ya instalado.


In [ ]:
import sys
import asyncio
import warnings
import requests
import time
import random
import csv
import threading
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeout
from playwright_stealth import Stealth

# FIX Windows/Python 3.14: forzar ProactorEventLoop antes de que Playwright lo use
if sys.platform == "win32":
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    print("✅ ProactorEventLoop activado")

print("Imports OK")

✅ ProactorEventLoop activado
Imports OK


### Paso 1: Scraping del listado

La página de listado (`seccion.asp`) no activa el challenge AWS WAF.

In [ ]:
URL_TARGET = "https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
}

publicaciones = []
print(f"Descargando listado: {URL_TARGET}...")
try:
    response = requests.get(URL_TARGET, headers=headers, timeout=15)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        items = soup.select(".item.anuncio")
        print(f"  -> Encontrados {len(items)} anuncios.")
        for item in items:
            anuncio_id   = item.get("data-id", "")
            titulo_tag   = item.select_one(".anuncio_title")
            titulo       = titulo_tag.get_text(strip=True) if titulo_tag else ""
            detalles_tag = item.select_one(".anuncio_details")
            aptitud      = detalles_tag.get_text(strip=True) if detalles_tag else ""
            ubi_tag      = item.select_one(".anuncio-distancia span")
            ubicacion    = ubi_tag.get_text(strip=True) if ubi_tag else ""
            user_tag     = item.select_one(".anuncio_username span")
            vendedor     = user_tag.get_text(strip=True) if user_tag else ""
            precio_tag   = item.select_one(".anuncio_price")
            precio_texto = precio_tag.get_text(strip=True) if precio_tag else ""
            pw_tag       = item.select_one(".precio_wrapper")
            precio_valor = pw_tag.get("data-price", "") if pw_tag else ""
            moneda       = pw_tag.get("data-currency", "") if pw_tag else ""
            condiciones  = " | ".join(
                c.get_text(strip=True) for c in item.select(".sdp_main_condition-text")
            )
            a_tag = item.find("a")
            href  = a_tag.get("href", "") if a_tag else ""
            if href.startswith("/"):
                link = f"https://www.agroads.com.ar{href}"
            elif href.startswith("http"):
                link = href
            else:
                link = f"https://www.agroads.com.ar/{href}" if href else ""
            publicaciones.append({
                "id": anuncio_id, "titulo": titulo, "aptitud": aptitud,
                "ubicacion": ubicacion, "vendedor": vendedor,
                "precio_texto": precio_texto, "precio_valor": precio_valor,
                "moneda": moneda, "condiciones": condiciones, "link": link,
            })
    else:
        print(f"  -> Error HTTP {response.status_code}")
except Exception as e:
    print(f"  -> Excepción: {type(e).__name__}: {e}")

print(f"Total recolectados: {len(publicaciones)}")

Descargando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36...
  -> Encontrados 60 anuncios.
Total recolectados: 60


### Paso 2: Crawler con Playwright + Stealth + Detección de CAPTCHA

**Lógica de detección de CAPTCHA (corregida):**  
Se activa **solo** cuando se cumplen las dos condiciones a la vez:  
1. El contenido real del anuncio (`h1`, `dl.grupo`) **NO** está presente.
2. Hay un marcador específico del challenge AWS WAF (`#captcha-container`, `awsWafIntegration`, o `aws-waf-token` en el HTML).

Esto evita falsos positivos en páginas normales que puedan tener elementos con 'captcha' en el ID.

In [ ]:
def _comportamiento_humano(page):
    """Simula scroll y movimiento de mouse para reducir huella de bot."""
    try:
        page.evaluate(f"window.scrollBy(0, {random.randint(200, 600)})")
        time.sleep(random.uniform(0.3, 0.8))
        page.mouse.move(random.randint(100, 900), random.randint(100, 600))
        time.sleep(random.uniform(0.2, 0.5))
    except Exception:
        pass


def _es_captcha_interactivo(page) -> bool:
    """
    Detecta CAPTCHA interactivo de AWS WAF usando dos condiciones simultáneas:
    1. El contenido real del anuncio (h1 o dl.grupo) NO está presente.
    2. Hay una señal explícita del sistema AWS WAF en el HTML.

    Esto evita falsos positivos: si el anuncio cargó bien, no se dispara
    aunque la página tenga elementos con 'captcha' en el ID (tokens, etc.).
    """
    # Condición 1: el contenido real del anuncio debe estar AUSENTE
    contenido_real_presente = bool(
        page.query_selector("h1") or page.query_selector("dl.grupo")
    )
    if contenido_real_presente:
        return False  # La página cargó bien, no hay CAPTCHA bloqueando

    # Condición 2: señales específicas de AWS WAF CAPTCHA interactivo
    try:
        html = page.content()
        señales_waf = [
            "AwsWafIntegration",    # JS del challenge AWS WAF
            "awsWafCookieDomainList",  # variable del challenge JS
            "aws-waf-token",        # cookie/token del WAF
            "challenge-container",  # ID del contenedor del challenge
            "amzn-captcha",        # clase específica de Amazon CAPTCHA
        ]
        if any(señal in html for señal in señales_waf):
            return True
    except Exception:
        pass

    return False


def _esperar_resolucion_captcha(page, index: int, total: int):
    """Pausa el crawler y espera a que el usuario resuelva el CAPTCHA."""
    print(f"\n{'='*60}")
    print(f"🚨 CAPTCHA DETECTADO en anuncio [{index}/{total}]")
    print("   1. Mirá la ventana de Chromium que está abierta.")
    print("   2. Resolvé el desafío manualmente.")
    print("   3. El crawler retomará automáticamente al terminar.")
    print(f"{'='*60}\n")
    max_espera, intervalo, transcurrido = 180, 2, 0
    while transcurrido < max_espera:
        time.sleep(intervalo)
        transcurrido += intervalo
        if not _es_captcha_interactivo(page):
            print("✅ CAPTCHA resuelto. Retomando crawl...\n")
            time.sleep(2)
            return
        if transcurrido % 30 == 0:
            print(f"   ⏳ Esperando... ({transcurrido}s / {max_espera}s)")
    print("⚠️  Tiempo agotado. Continuando de todas formas...")


def _extraer_detalle(page, url: str, index: int, total: int) -> dict:
    """Navega al detalle de un anuncio, maneja WAF/CAPTCHA y extrae datos."""
    detalle = {
        "superficie_ha": "", "transaccion": "",
        "instalaciones": "", "descripcion_completa": "",
    }
    try:
        page.goto(url, wait_until="domcontentloaded", timeout=30000)
        time.sleep(1.5)  # pausa para que el JS del WAF se evalúe

        # Solo pausar si el CAPTCHA realmente bloquea el contenido
        if _es_captcha_interactivo(page):
            _esperar_resolucion_captcha(page, index, total)

        page.wait_for_selector("h1, dl.grupo", timeout=20000)
        _comportamiento_humano(page)

        soup = BeautifulSoup(page.content(), "html.parser")

        for dl in soup.find_all("dl", class_="grupo"):
            dts = [t.get_text(strip=True) for t in dl.find_all("dt")]
            dds = [t.get_text(strip=True) for t in dl.find_all("dd")]
            for etiqueta, valor in zip(dts, dds):
                et = etiqueta.lower()
                if "superficie" in et:
                    detalle["superficie_ha"] = valor
                elif "transacci" in et:
                    detalle["transaccion"] = valor
                elif "instalacion" in et:
                    detalle["instalaciones"] = valor

        desc_dl = soup.find("dl", class_=lambda c: c and "descripcion" in c)
        if desc_dl:
            dd = desc_dl.find("dd")
            if dd:
                detalle["descripcion_completa"] = dd.get_text(" ", strip=True)

    except PlaywrightTimeout:
        print(f"      [TIMEOUT] {url}")
    except Exception as exc:
        print(f"      [ERROR] {type(exc).__name__}: {exc}")

    return detalle


def _playwright_worker(pubs, headless, delay_min, delay_max, resultado, errores):
    """Worker de Playwright en thread separado."""
    try:
        with Stealth().use_sync(sync_playwright()) as pw:
            browser = pw.chromium.launch(
                headless=headless,
                args=[
                    "--no-sandbox",
                    "--disable-blink-features=AutomationControlled",
                    "--disable-infobars",
                    "--window-size=1280,800",
                ],
            )
            ctx = browser.new_context(
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/122.0.0.0 Safari/537.36"
                ),
                locale="es-AR",
                timezone_id="America/Argentina/Cordoba",
                viewport={"width": 1280, "height": 800},
                extra_http_headers={
                    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
                    "Referer": URL_TARGET,
                },
            )

            page = ctx.new_page()

            print("[INIT] Estableciendo cookies de sesión en el listado...")
            page.goto(URL_TARGET, wait_until="domcontentloaded", timeout=30000)
            _comportamiento_humano(page)
            time.sleep(random.uniform(2, 4))

            datos = []
            total = len(pubs)
            for i, pub in enumerate(pubs, 1):
                url = pub.get("link", "")
                print(f"[{i}/{total}] {url}")

                fila = {
                    **pub,
                    "superficie_ha": "", "transaccion": "",
                    "instalaciones": "", "descripcion_completa": "",
                }
                if url:
                    fila.update(_extraer_detalle(page, url, i, total))

                datos.append(fila)

                if i % 10 == 0:
                    pausa = random.uniform(15, 25)
                    print(f"   -> Pausa larga cada 10 anuncios: {pausa:.1f}s")
                else:
                    pausa = random.uniform(delay_min, delay_max)
                    print(f"   -> pausa {pausa:.1f}s")
                time.sleep(pausa)

            browser.close()
            resultado.extend(datos)
            print(f"\n✅ Extracción finalizada: {len(datos)} registros.")

    except Exception as exc:
        msg = f"{type(exc).__name__}: {exc}"
        print(f"[WORKER ERROR] {msg}")
        errores.append(msg)


def crawl_detalles(publicaciones, headless=False, delay_min=5.0, delay_max=12.0):
    """
    Crawl completo con stealth, comportamiento humano y manejo de CAPTCHA.

    Args:
        publicaciones : lista de dicts con clave 'link'.
        headless      : False = navegador visible (necesario para resolver CAPTCHAs).
        delay_min/max : pausa aleatoria entre peticiones (segundos).
    """
    if headless:
        print("⚠️  headless=True: si aparece CAPTCHA no podrás verlo ni resolverlo.")
        print("   Recomendado: headless=False para el crawl completo.\n")

    resultado, errores = [], []
    hilo = threading.Thread(
        target=_playwright_worker,
        args=(publicaciones, headless, delay_min, delay_max, resultado, errores),
        daemon=True,
    )
    hilo.start()
    hilo.join()

    if errores:
        print(f"\n⚠️  Errores en el worker: {errores}")
    return resultado


print("Funciones del crawler definidas correctamente.")

Funciones del crawler definidas correctamente.


### Paso 3: Ejecutar el crawler

> **`headless=False`** es obligatorio para poder resolver CAPTCHAs manualmente.
> Cuando aparezca uno, el código imprimirá las instrucciones y esperará hasta que lo resuelvas.

In [ ]:
datos_completos = crawl_detalles(
    publicaciones=publicaciones,    # lista completa (60 anuncios)
    headless=False,                 # VISIBLE: necesario para resolver CAPTCHAs
    delay_min=5.0,
    delay_max=12.0,
)

if datos_completos:
    print("\n--- Primer registro obtenido ---")
    for k, v in datos_completos[0].items():
        print(f"  {k:20s}: {str(v)[:80]}")

[INIT] Estableciendo cookies de sesión en el listado...
[1/60] https://www.agroads.com.ar/detalle.asp?clasi=1092607
   -> pausa 6.7s
[2/60] https://www.agroads.com.ar/detalle.asp?clasi=1052572
      [ERROR] Error: Page.goto: net::ERR_ABORTED at https://www.agroads.com.ar/detalle.asp?clasi=1052572
Call log:
  - navigating to "https://www.agroads.com.ar/detalle.asp?clasi=1052572", waiting until "domcontentloaded"

   -> pausa 11.3s
[3/60] https://www.agroads.com.ar/detalle.asp?clasi=1091956
   -> pausa 6.1s
[4/60] https://www.agroads.com.ar/detalle.asp?clasi=1024403
   -> pausa 10.2s
[5/60] https://www.agroads.com.ar/detalle.asp?clasi=1090947
   -> pausa 9.3s
[6/60] https://www.agroads.com.ar/detalle.asp?clasi=1090780
   -> pausa 11.4s
[7/60] https://www.agroads.com.ar/detalle.asp?clasi=1090452
   -> pausa 9.5s
[8/60] https://www.agroads.com.ar/detalle.asp?clasi=1068709
   -> pausa 7.0s
[9/60] https://www.agroads.com.ar/detalle.asp?clasi=1085981
   -> pausa 9.1s
[10/60] https://www.agroa

In [ ]:
archivo_salida = "anuncios_agroads_cordoba.csv"

if datos_completos:
    columnas = datos_completos[0].keys()
    with open(archivo_salida, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columnas)
        writer.writeheader()
        writer.writerows(datos_completos)
    print(f"✅ Se guardaron {len(datos_completos)} registros en '{archivo_salida}'.")
else:
    print("⚠️  No hay datos para guardar.")

✅ Se guardaron 60 registros en 'anuncios_agroads_cordoba.csv'.
